# Quoting Engine

A market maker quotes a bid ask based on a spread centered on the fair value point from heston.

The order arrivals will be modeled with a poisson process. 

The Avellaneda-Stoikov framework is used to derive optimal bid/ask quotes by computing a reservation price adjusted for inventory risk, then placing a spread symmetrically around it.

Outputs: bid/ask quotes over time, inventory path, and P&L attribution.

In [ ]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [ ]:
# grab r0 from vasicek params

with open("../data/vasicek_params.json") as f:
    params = json.load(f)

r0 = params['r0']

# market snapshot - April 24th, 2026 close
S0 = 86.71 # TLT spot price
fair_value = 2.50 # placeholder Heston call price
sigma = 0.3 # placeholder implied vol

print(f"r0: {r0}")
print(f"S0: {S0}")
print(f"fair_value: {fair_value}")
print(f"sigma: {sigma}")

### Avellaneda-Stoikov

Academic standard for options market making. The core idea: you don't quote symmetrically around fair value - you skew your quotes based on your inventory to manage risk.

Three formulas:
$$r_{price} = \text{fair\_value} - \text{inventory} \cdot \gamma \cdot \sigma^2 \cdot (T - t)$$

$$\text{spread} = \gamma \cdot \sigma^2 \cdot (T-t) + \frac{2}{\gamma} \ln\left(1 + \frac{\gamma}{\kappa_{MM}}\right)$$

$$\text{bid} = r_{price} - \frac{\text{spread}}{2}, \quad \text{ask} = r_{price} + \frac{\text{spread}}{2}$$

Parameters:
* $\gamma$      : Our risk aversion. Higher = wider spreads, more aggressive inventory skew
* $\kappa_{MM}$ : Rate at which orders arrive. Higher = tighter spreads (more flow = less need to widen)
* $T$           : Expiration of the option
* $t$           : current time (start at 0)
* inventory     : Start at zero

In [ ]:
# parameters
T = 21/365
t = 0.0
gamma = 0.1
kappa_MM = 20   # 20 orders per year
inventory = 0

# calculations
r_price = fair_value - inventory * gamma * sigma**2 * (T-t)

spread = gamma * sigma**2 * (T-t) + (2/gamma) * np.log(1 + (gamma/kappa_MM))

bid = r_price - (spread/2)
ask = r_price + (spread/2)

print(r_price, spread,bid, ask)


### Poisson Order Arrival Simulation
We simulate order flow over the 21-day life of the option. At each daily time step, buyers and sellers arrive randomly via independent Poisson processes with arrival rate `kappa_MM`. Not every arrival results in a fill — fill probability decays exponentially with distance of the quote from fair value. This captures adverse selection: the further our quote is from fair value, the less likely we get hit.

We track inventory, P&L, and re-calculate reservation price and spread at each step to reflect inventory skew.

In [ ]:
# initialize tracking variables
inventory = 0          # current net position (+ long, - short)
bids_filled = 0        # number of times sellers hit our bid
asks_filled = 0        # number of times buyers hit our ask
pnl = 0                # cumulative cash P&L
inventory_track = []   # daily inventory snapshots for plotting
pnl_track = []         # daily total P&L snapshots for plotting

# risk limit — max contracts long or short before we pull quotes on that side
max_inventory = 3

# simulation — one iteration per trading day
for day in range(21):
    t = day / 365  # current time in years

    # recalculate quotes based on current inventory
    # A-S reservation price skews away from fair value as inventory builds
    r_price = fair_value - inventory * gamma * sigma**2 * (T - t)
    spread = gamma * sigma**2 * (T - t) + (2/gamma) * np.log(1 + (gamma/kappa_MM))
    bid = r_price - (spread/2)
    ask = r_price + (spread/2)

    # Poisson arrivals — kappa_MM is annual rate, scaled to daily
    buyers = np.random.poisson(kappa_MM / 365)
    sellers = np.random.poisson(kappa_MM / 365)

    # fill probability — exponential decay with distance from fair value
    # captures adverse selection: quotes far from fair value rarely fill
    prob_bid = np.exp(-kappa_MM * (fair_value - bid))
    prob_ask = np.exp(-kappa_MM * (ask - fair_value))

    # ask side — only quote if not at short inventory limit
    # two conditions: at least one buyer arrived AND uniform draw beats fill probability
    if inventory > -max_inventory:
        if buyers > 0 and np.random.rand() < prob_ask:
            inventory -= 1    # we sold, inventory decreases
            asks_filled += 1
            pnl += ask        # cash inflow at ask price

    # bid side — only quote if not at long inventory limit
    if inventory < max_inventory:
        if sellers > 0 and np.random.rand() < prob_bid:
            inventory += 1    # we bought, inventory increases
            bids_filled += 1
            pnl -= bid        # cash outflow at bid price

    # record end of day state
    inventory_track.append(inventory)
    pnl_track.append(pnl + inventory * fair_value)  # cash P&L + mark-to-market

# mark-to-market P&L — value of remaining inventory at fair value
mtm_pnl = inventory * fair_value
total_pnl = pnl + mtm_pnl

# tracking dataframe
tracking_df = pd.DataFrame({'Inventory': inventory_track, 'Total P&L': pnl_track})

print(f"Final inventory: {inventory}")
print(f"Bids filled: {bids_filled}")
print(f"Asks filled: {asks_filled}")
print(f"Cash P&L: {pnl:.4f}")
print(f"Mark-to-market P&L: {mtm_pnl:.4f}")
print(f"Total P&L: {total_pnl:.4f}")

# plot results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

ax1.plot(range(21), inventory_track)
ax1.axhline(0, color='red', linestyle='--', linewidth=0.8)
ax1.set_title('Inventory Over Time')
ax1.set_xlabel('Day')
ax1.set_ylabel('Inventory')
ax1.set_xticks(range(21))
ax1.grid(True, alpha=0.3)

ax2.plot(range(21), pnl_track, color='green')
ax2.axhline(0, color='red', linestyle='--', linewidth=0.8)
ax2.set_title('Total P&L Over Time')
ax2.set_xlabel('Day')
ax2.set_ylabel('P&L ($)')
ax2.set_xticks(range(21))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()